In [2]:
import pandas as pd

df = pd.read_csv("breast-cancer.csv")    # Your Kaggle dataset

# Remove unnecessary columns
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

# Convert target to numeric
df["diagnosis"] = df["diagnosis"].map({"M":1, "B":0})

X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

In [8]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=4)

X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features.tolist())

Selected Features:
['concave points_mean', 'radius_worst', 'perimeter_worst', 'concave points_worst']


In [9]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

selector = RFE(model, n_features_to_select=10)

selector.fit(X, y)

selected = X.columns[selector.support_]

print(selected.tolist())

['perimeter_mean', 'area_mean', 'concavity_mean', 'concave points_mean', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'concavity_worst', 'concave points_worst']


In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_,
                       index=X.columns)

importance = importance.sort_values(ascending=False)

print(importance)

area_worst                 0.139357
concave points_worst       0.132225
concave points_mean        0.107046
radius_worst               0.082848
perimeter_worst            0.080850
perimeter_mean             0.067990
concavity_mean             0.066917
area_mean                  0.060462
concavity_worst            0.037339
radius_mean                0.034843
area_se                    0.029553
compactness_worst          0.019864
texture_worst              0.017485
texture_mean               0.015225
radius_se                  0.014264
smoothness_worst           0.012232
compactness_mean           0.011597
perimeter_se               0.010085
symmetry_worst             0.008179
smoothness_mean            0.007958
fractal_dimension_se       0.005942
concavity_se               0.005820
compactness_se             0.005612
smoothness_se              0.004722
fractal_dimension_worst    0.004497
concave points_se          0.003760
texture_se                 0.003744
symmetry_se                0

In [6]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.1,
    random_state=42
)

lasso.fit(X_scaled, y)

selector = SelectFromModel(lasso, prefit=True)

selected = X.columns[selector.get_support()]

print(selected.tolist())

['concave points_mean', 'radius_se', 'radius_worst', 'texture_worst', 'smoothness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst']


d:\ProjectX\ML\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\ProjectX\ML\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [7]:
import numpy as np

corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]

print("Features Removed:")
print(to_drop)

X_selected = X.drop(columns=to_drop)

print("Remaining Features:")
print(X_selected.columns.tolist())

Features Removed:
['perimeter_mean', 'area_mean', 'concave points_mean', 'perimeter_se', 'area_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'concave points_worst']
Remaining Features:
['radius_mean', 'texture_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'symmetry_worst', 'fractal_dimension_worst']
